# IRD Phenotype Clustering — Synthetic Data Demonstration

> ⚠️ **Synthetic data demonstration only.**
> All gene names, HPO term identifiers, and numeric values are entirely fabricated.
> No real patient, clinical, or unpublished research data are present.

This notebook demonstrates an HPO-based phenotypic similarity clustering pipeline for
Inherited Retinal Disease (IRD) genes. The pipeline mirrors the analytical *structure* of
a real research pipeline: IC-based phenotype filtering → Lin's similarity / BMA → kNN /
threshold graph construction → Leiden community detection → perturbation stability →
Fisher's exact test enrichment (BH FDR-corrected).

All numeric thresholds are **illustrative demo values chosen for this synthetic dataset**
and do not reflect the calibrated parameters of the real unpublished pipeline.


## § 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


## ⚙ Demo Configuration

All parameters below are chosen for this synthetic demonstration only.
They are **not** the calibrated thresholds of the real research pipeline.

In a standalone script these would be exposed as `argparse` CLI arguments
(consistent with other projects in this portfolio).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO CONFIGURATION — illustrative values only; real pipeline uses different ones
# ─────────────────────────────────────────────────────────────────────────────
RANDOM_SEED = 0       # reproducibility seed (not the real pipeline's seed)

# § 2 — Phenotype filtering
MIN_IC               = 4.0   # information-content floor
MIN_DEPTH            = 5.0   # ontology-depth proxy floor
MAX_GENE_FREQ        = 0.20  # max fraction of genes a term may annotate
MIN_SUPPORTING_GENES = 2     # minimum genes required per surviving HPO term

# § 4 — Graph construction
KNN_K_VALUES         = [3, 6, 9]        # k-NN graph variants (illustrative round sweep)
THRESHOLD_PCTILES    = [60, 75, 90]     # similarity-threshold graph percentile variants

# § 5 — Leiden community detection
LEIDEN_RESOLUTION    = 1.0   # partition resolution parameter

# § 6 — Perturbation stability
NUM_RUNS             = 8     # perturbation iterations (also reduced from real for demo speed)
NOISE_SD             = 0.05  # Gaussian noise σ applied to similarity matrix
CORE_THR             = 0.65  # co-clustering frequency → core gene
PERI_THR             = 0.35  # co-clustering frequency → peripheral gene

# § 7 — Module QC
MIN_MODULE_SIZE      = 4     # modules smaller than this are excluded from enrichment
LARGE_MODULE_SIZE    = 15    # demo-scale "large module" threshold (real pipeline: much larger)

# § 8 — Phenotypic signatures
SIGNATURE_MIN_FREQ   = 0.30  # min fraction of module genes with HPO term (pre-filter)
FDR_ALPHA            = 0.05  # BH-FDR significance threshold
                              # (standard statistical convention — not pipeline-specific)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(RANDOM_SEED)
OUT_DIR = Path("../outputs/demo_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded. Output directory:", OUT_DIR.resolve())


## § 1 — Load Synthetic Gene–HPO Matrix

The real pipeline begins with a binary gene × HPO term annotation matrix
(hundreds of genes, thousands of HPO terms). After IC-based filtering the
working matrix is substantially smaller.

Here we load a small synthetic stand-in (40 genes × 60 HPO terms) stored in
`../data/synthetic_gene_hpo_matrix.csv`. Gene names follow the pattern
`GENE_A01…GENE_D10`; HPO term IDs use the prefix `HP:SYNTH_` (clearly non-real).

**Synthetic data structure:** genes are divided into four groups (A–D, 10 genes each),
each with a private block of 15 HPO terms shared at ~55 % within the group
(2 % background across other groups). This mimics biologically coherent modules
while keeping all values fabricated.


In [ ]:
df_raw = pd.read_csv("../data/synthetic_gene_hpo_matrix.csv", index_col="gene")
genes_raw = df_raw.index.tolist()
n_genes_raw = len(genes_raw)

print(f"Raw matrix: {df_raw.shape[0]} genes × {df_raw.shape[1]} HPO terms")
print(f"Genes (first 5): {genes_raw[:5]} …")
print(f"HPO terms (first 5): {df_raw.columns.tolist()[:5]} …")
df_raw.iloc[:4, :8]


## § 2 — Phenotype Filtering (IC-based)

The real pipeline applies multi-stage filters to retain informative, specific HPO
terms and remove genes that lose all annotated terms. The filters use:

| Filter | Removes |
|--------|---------|
| Information Content (IC) floor | Non-specific, ubiquitous terms |
| Ontology depth floor | Root-level terms with little discriminatory power |
| Max gene frequency | Terms shared by too many genes to be module-specific |
| Min supporting genes | Terms too rare to support statistical inference |

**Demo parameters** (illustrative — not the real pipeline's calibrated values):
- `MIN_IC = 4.0`, `MIN_DEPTH = 5.0`, `MAX_GENE_FREQ = 0.20`, `MIN_SUPPORTING_GENES = 2`

In the real pipeline, **IC** is computed from the HPO annotation database via
`pyhpo`: IC(term) = −log₂(p(term)), where p is the annotated-gene proportion
across all descendants. Here IC and depth are **synthetically assigned** (independent
random draws) to keep the demo self-contained.


In [ ]:
np.random.seed(RANDOM_SEED)
n_terms_raw = df_raw.shape[1]

# Synthetic IC values: Uniform(2.0, 8.0) — representative range, not real values
ic_values = pd.Series(
    np.random.uniform(2.0, 8.0, n_terms_raw),
    index=df_raw.columns
)

# Synthetic depth values: independent draw Uniform(1, 10)
# (in the real ontology depth correlates with IC, but they are structurally distinct)
np.random.seed(RANDOM_SEED + 1)
depth_values = pd.Series(
    np.random.uniform(1.0, 10.0, n_terms_raw),
    index=df_raw.columns
)

gene_freq    = df_raw.mean(axis=0)           # fraction of genes annotated per term
gene_counts  = df_raw.sum(axis=0)            # absolute count per term

mask_ic    = ic_values >= MIN_IC
mask_depth = depth_values >= MIN_DEPTH
mask_freq  = gene_freq <= MAX_GENE_FREQ
mask_min   = gene_counts >= MIN_SUPPORTING_GENES

keep_terms = mask_ic & mask_depth & mask_freq & mask_min
df_filt    = df_raw.loc[:, keep_terms]

# Drop genes that lost all annotated terms
gene_mask = df_filt.sum(axis=1) > 0
df_filt   = df_filt.loc[gene_mask]

print(f"After filtering:")
print(f"  {df_filt.shape[0]} genes  (removed {n_genes_raw - df_filt.shape[0]})")
print(f"  {df_filt.shape[1]} HPO terms  (removed {n_terms_raw - df_filt.shape[1]})")
print(f"  Removed breakdown: IC<{MIN_IC}: {(~mask_ic).sum()}, "
      f"depth<{MIN_DEPTH}: {(~mask_depth).sum()}, "
      f"freq>{MAX_GENE_FREQ}: {(~mask_freq).sum()}, "
      f"genes<{MIN_SUPPORTING_GENES}: {(~mask_min).sum()}")


## § 3 — Lin's Similarity (Proxy) & BMA Gene Similarity

### Real pipeline: Lin's similarity + BMA
The real pipeline uses **Lin's IC-based similarity** (Lin 1998) for term–term similarity:

$$\text{Lin}(A, B) = \frac{2 \cdot IC(\text{MICA}(A,B))}{IC(A) + IC(B)}$$

where MICA is the Most Informative Common Ancestor in the HPO DAG, computed via
`pyhpo`. This requires traversing the full ontology structure.

**Demo stand-in:** Jaccard similarity between term co-annotation vectors, which has the
same [0, 1] range and encodes co-occurrence structure without requiring `hp.obo`.

**BMA (Best Match Average)** is then applied exactly as in the real pipeline:

$$\text{BMA}(G_i, G_j) = \frac{\sum_{a \in G_i} \max_{b \in G_j} \text{sim}(a,b)
+ \sum_{b \in G_j} \max_{a \in G_i} \text{sim}(a,b)}{|G_i| + |G_j|}$$


In [ ]:
np.random.seed(RANDOM_SEED)

terms = df_filt.columns.tolist()
genes = df_filt.index.tolist()
M     = df_filt.values.astype(float)   # genes × terms
n_t   = len(terms)
n_g   = len(genes)
T     = M.T                            # terms × genes

# ── Term–term Jaccard similarity (Lin proxy) ──────────────────────────────────
print("Computing term–term Jaccard similarity …")
term_sim = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(i, n_t):
        inter = np.dot(T[i], T[j])
        union = np.sum((T[i] + T[j]) > 0)
        val   = inter / union if union > 0 else 0.0
        term_sim[i, j] = term_sim[j, i] = val

# ── BMA gene–gene similarity ──────────────────────────────────────────────────
print("Computing BMA gene–gene similarity …")
gene_sim = np.zeros((n_g, n_g))
for i in range(n_g):
    ti = np.where(M[i] > 0)[0]
    for j in range(i, n_g):
        tj = np.where(M[j] > 0)[0]
        if len(ti) == 0 or len(tj) == 0:
            gene_sim[i, j] = gene_sim[j, i] = 0.0
            continue
        fwd = sum(term_sim[a, tj].max() for a in ti)
        rev = sum(term_sim[b, ti].max() for b in tj)
        bma = (fwd + rev) / (len(ti) + len(tj))
        gene_sim[i, j] = gene_sim[j, i] = bma

df_sim = pd.DataFrame(gene_sim, index=genes, columns=genes)
print(f"Gene–gene BMA: mean={gene_sim.mean():.4f}, median={np.median(gene_sim):.4f}")

df_sim.to_csv(OUT_DIR / "gene_similarity_matrix_demo.csv")
print("Saved gene_similarity_matrix_demo.csv")


In [ ]:
# ── Similarity distribution ───────────────────────────────────────────────────
upper = gene_sim[np.triu_indices(n_g, k=1)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(upper, bins=30, color="#5C6BC0", edgecolor="white", linewidth=0.5)
ax.axvline(upper.mean(), color="#E53935", linestyle="--", linewidth=1.5,
           label=f"Mean = {upper.mean():.3f}")
ax.set_xlabel("BMA Similarity Score", fontsize=12)
ax.set_ylabel("Gene-pair Count", fontsize=12)
ax.set_title("Gene–Gene BMA Similarity Distribution\n"
             "(Synthetic data – demonstration only)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "similarity_distribution.png", dpi=150)
plt.show()
print("Saved similarity_distribution.png")


## § 4 — Graph Construction (kNN + Threshold Variants)

The real pipeline systematically sweeps multiple graph construction strategies
(kNN and similarity-threshold graphs) to explore community structure space.

Here we build **6 representative variants** using the demo parameters
`KNN_K_VALUES = [3, 6, 9]` and `THRESHOLD_PCTILES = [60, 75, 90]`.
The selection logic applied in § 5 is identical to the real pipeline.


In [ ]:
def build_knn_graph(sim_matrix, genes, k):
    """kNN graph: each node keeps edges to its k most similar neighbours."""
    G = nx.Graph()
    G.add_nodes_from(genes)
    n = len(genes)
    for i in range(n):
        row = sim_matrix[i].copy()
        row[i] = -1
        top_k = np.argsort(row)[-k:]
        for j in top_k:
            if sim_matrix[i, j] > 0:
                G.add_edge(genes[i], genes[j], weight=float(sim_matrix[i, j]))
    return G

def build_threshold_graph(sim_matrix, genes, pctile):
    """Threshold graph: keep edges where similarity >= given percentile."""
    upper = sim_matrix[np.triu_indices(len(genes), k=1)]
    threshold = np.percentile(upper, pctile)
    G = nx.Graph()
    G.add_nodes_from(genes)
    n = len(genes)
    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= threshold:
                G.add_edge(genes[i], genes[j], weight=float(sim_matrix[i, j]))
    return G

graphs = {}
for k in KNN_K_VALUES:
    graphs[f"knn_{k}"] = build_knn_graph(gene_sim, genes, k)
for p in THRESHOLD_PCTILES:
    graphs[f"thr_{p}"] = build_threshold_graph(gene_sim, genes, p)

for name, G in graphs.items():
    print(f"  {name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## § 5 — Leiden Community Detection & Graph Selection

Each graph variant is clustered with the **Leiden algorithm**
(`leidenalg.ModularityVertexPartition`, `LEIDEN_RESOLUTION = 1.0`, `seed = RANDOM_SEED`).
The best graph is selected by:

$$\text{score} = \text{modularity} \times \frac{\text{mean within-community similarity}}
{\text{mean between-community similarity}}$$

This jointly rewards global community structure (modularity) and within-module
phenotypic coherence relative to the background (cohesion/separation ratio).


In [ ]:
def nx_to_igraph(G_nx, genes):
    gene_idx = {g: i for i, g in enumerate(genes)}
    edges    = [(gene_idx[u], gene_idx[v]) for u, v in G_nx.edges()]
    weights  = [G_nx[u][v]["weight"] for u, v in G_nx.edges()]
    ig_g     = ig.Graph(n=len(genes), edges=edges, directed=False)
    ig_g.es["weight"] = weights
    ig_g.vs["name"]   = genes
    return ig_g

results = {}
for name, G_nx in graphs.items():
    ig_g      = nx_to_igraph(G_nx, genes)
    partition = leidenalg.find_partition(
        ig_g, leidenalg.ModularityVertexPartition,
        weights="weight", seed=RANDOM_SEED
    )
    modularity = partition.modularity
    labels     = np.array(partition.membership)

    within, between, n_w, n_b = 0.0, 0.0, 0, 0
    for i in range(n_g):
        for j in range(i + 1, n_g):
            if labels[i] == labels[j]:
                within += gene_sim[i, j]; n_w += 1
            else:
                between += gene_sim[i, j]; n_b += 1
    coh_sep = (within / n_w) / (between / n_b) if n_w > 0 and n_b > 0 else 1.0
    score   = modularity * coh_sep

    results[name] = {
        "modularity": modularity, "coh_sep": coh_sep,
        "score": score, "n_communities": len(set(labels)), "labels": labels
    }
    print(f"  {name}: mod={modularity:.4f}, coh/sep={coh_sep:.4f}, "
          f"score={score:.4f}, n_communities={len(set(labels))}")

best_name      = max(results, key=lambda k: results[k]["score"])
best           = results[best_name]
community_labels = best["labels"]
print(f"\n✓ Best graph: {best_name}  "
      f"(score={best['score']:.4f}, communities={best['n_communities']})")


## § 6 — Perturbation-Based Stability Analysis

The real pipeline assesses stability by adding Gaussian noise to the gene–gene
similarity matrix and re-clustering, recording how often each gene pair ends up in
the same community. Genes are classified by their mean co-clustering frequency
with module-mates:

| Class | Co-clustering frequency |
|-------|------------------------|
| **Core** | ≥ `CORE_THR` |
| **Peripheral** | `PERI_THR` – `CORE_THR` |
| **Unstable** | < `PERI_THR` |

**Demo parameters** (illustrative): `NUM_RUNS = 8`, `NOISE_SD = 0.05`,
`CORE_THR = 0.65`, `PERI_THR = 0.35`.


In [ ]:
np.random.seed(RANDOM_SEED)
co_cluster_count = np.zeros((n_g, n_g))

for run in range(NUM_RUNS):
    noise         = np.random.normal(0, NOISE_SD, gene_sim.shape)
    noise         = (noise + noise.T) / 2
    sim_perturbed = np.clip(gene_sim + noise, 0, 1)
    np.fill_diagonal(sim_perturbed, 0)

    if best_name.startswith("knn"):
        k_val  = int(best_name.split("_")[1])
        G_p    = build_knn_graph(sim_perturbed, genes, k_val)
    else:
        p_val  = int(best_name.split("_")[1])
        G_p    = build_threshold_graph(sim_perturbed, genes, p_val)

    ig_p    = nx_to_igraph(G_p, genes)
    part_p  = leidenalg.find_partition(
        ig_p, leidenalg.ModularityVertexPartition, weights="weight", seed=run
    )
    lbls = np.array(part_p.membership)
    for i in range(n_g):
        for j in range(i + 1, n_g):
            if lbls[i] == lbls[j]:
                co_cluster_count[i, j] += 1
                co_cluster_count[j, i] += 1

stability_matrix = co_cluster_count / NUM_RUNS

gene_stability = np.zeros(n_g)
for i in range(n_g):
    mates = np.where(community_labels == community_labels[i])[0]
    mates = mates[mates != i]
    if len(mates) > 0:
        gene_stability[i] = stability_matrix[i, mates].mean()

gene_class = pd.Series(index=genes, dtype=str)
gene_class[gene_stability >= CORE_THR]                                    = "Core"
gene_class[(gene_stability >= PERI_THR) & (gene_stability < CORE_THR)]   = "Peripheral"
gene_class[gene_stability < PERI_THR]                                     = "Unstable"

print("Gene classification:")
print(gene_class.value_counts().to_string())


In [ ]:
# ── Stability heatmap ─────────────────────────────────────────────────────────
order      = np.argsort(community_labels)
sm_ordered = stability_matrix[np.ix_(order, order)]
g_ordered  = [genes[i] for i in order]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sm_ordered, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Co-clustering frequency")

for b in np.where(np.diff(np.sort(community_labels)))[0] + 0.5:
    ax.axhline(b, color="black", linewidth=0.8)
    ax.axvline(b, color="black", linewidth=0.8)

ax.set_xticks(range(n_g)); ax.set_xticklabels(g_ordered, rotation=90, fontsize=6)
ax.set_yticks(range(n_g)); ax.set_yticklabels(g_ordered, fontsize=6)
ax.set_title("Gene Co-clustering Stability Matrix\n(Synthetic data – demonstration only)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "gene_stability_heatmap.png", dpi=150)
plt.show()
print("Saved gene_stability_heatmap.png")


## § 7 — Module QC & Size Distribution

Modules are evaluated against size thresholds:
- **Small** (< `MIN_MODULE_SIZE`): excluded from enrichment analysis
- **Large** (> `LARGE_MODULE_SIZE`): eligible for hierarchical sub-clustering

**Demo parameters**: `MIN_MODULE_SIZE = 4`, `LARGE_MODULE_SIZE = 15`.


In [ ]:
module_ids  = sorted(set(community_labels))
module_info = []
for mid in module_ids:
    idx   = np.where(community_labels == mid)[0]
    size  = len(idx)
    genes_in = [genes[i] for i in idx]
    if size < MIN_MODULE_SIZE:
        qc = "excluded_small"
    elif size > LARGE_MODULE_SIZE:
        qc = "large_subcluster_eligible"
    else:
        qc = "pass"
    module_info.append({"module_id": mid, "size": size,
                        "qc_status": qc, "genes": ", ".join(genes_in)})

df_module_qc = pd.DataFrame(module_info)
df_module_qc.to_csv(OUT_DIR / "module_QC_assignment_demo.csv", index=False)
print(df_module_qc[["module_id", "size", "qc_status"]].to_string(index=False))


In [ ]:
colors = {"pass": "#42A5F5",
          "excluded_small": "#EF5350",
          "large_subcluster_eligible": "#FFA726"}

fig, ax = plt.subplots(figsize=(7, 4))
for _, row in df_module_qc.iterrows():
    ax.bar(f"M{row['module_id']}", row["size"],
           color=colors.get(row["qc_status"], "#90A4AE"), edgecolor="white")
ax.axhline(MIN_MODULE_SIZE, color="#EF5350", linestyle="--", linewidth=1,
           label=f"Min size ({MIN_MODULE_SIZE})")
ax.axhline(LARGE_MODULE_SIZE, color="#FFA726", linestyle="--", linewidth=1,
           label=f"Large threshold ({LARGE_MODULE_SIZE})")
ax.set_xlabel("Module", fontsize=12)
ax.set_ylabel("Gene Count", fontsize=12)
ax.set_title("Module Size Distribution\n(Synthetic data – demonstration only)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(OUT_DIR / "module_size_distribution.png", dpi=150)
plt.show()
print("Saved module_size_distribution.png")


## § 8 — Phenotypic Signature Detection (Fisher's Exact Test + BH FDR)

For each module × HPO term pair the pipeline:

1. **Pre-filters** terms present in < `SIGNATURE_MIN_FREQ` of module genes
2. Runs a **one-sided Fisher's exact test** (alternative = 'greater')
3. Applies **Benjamini-Hochberg FDR** across all tested pairs

**Demo parameters**: `SIGNATURE_MIN_FREQ = 0.30`, `FDR_ALPHA = 0.05`
(FDR α = 0.05 is a universal statistical convention, not a pipeline-specific choice).


In [ ]:
np.random.seed(RANDOM_SEED)
n_bg         = df_filt.shape[0]
test_records = []

valid_modules = df_module_qc[df_module_qc["qc_status"] != "excluded_small"]["module_id"].tolist()

for mid in valid_modules:
    mod_idx    = np.where(community_labels == mid)[0]
    n_mod      = len(mod_idx)
    mod_matrix = M[np.ix_(mod_idx, range(M.shape[1]))]

    for ti, term in enumerate(df_filt.columns):
        in_mod = mod_matrix[:, ti].sum()
        if in_mod / n_mod < SIGNATURE_MIN_FREQ:
            continue

        not_in_mod = n_mod - in_mod
        in_bg_only = M[:, ti].sum() - in_mod
        not_bg     = n_bg - n_mod - in_bg_only

        table = [[in_mod, not_in_mod], [in_bg_only, not_bg]]
        try:
            _, pval = fisher_exact(table, alternative="greater")
        except Exception:
            pval = 1.0
        test_records.append({
            "module_id": mid, "hpo_term": term,
            "in_module": in_mod, "module_size": n_mod, "p_value": pval
        })

df_tests = pd.DataFrame(test_records)
print(f"Tests after pre-filtering: {len(df_tests)}")

if df_tests.empty:
    raise RuntimeError(
        "No tests passed the pre-filter. "
        "Check that the synthetic matrix has enough within-module term coverage."
    )

reject, qvals, _, _ = multipletests(df_tests["p_value"].values,
                                    alpha=FDR_ALPHA, method="fdr_bh")
df_tests["q_value"]    = qvals
df_tests["significant"] = reject

sig = df_tests[df_tests["significant"]]
print(f"FDR-significant signatures (q < {FDR_ALPHA}): {len(sig)}")
print("\nTop 10 signatures:")
print(sig.sort_values("q_value").head(10)[
    ["module_id", "hpo_term", "in_module", "p_value", "q_value"]
].to_string(index=False))


In [ ]:
# ── Module × HPO enrichment heatmap ──────────────────────────────────────────
top_terms = (sig.groupby("hpo_term")["in_module"].sum()
               .sort_values(ascending=False).head(20).index.tolist())

pivot_data = {}
for mid in valid_modules:
    row = {}
    for term in top_terms:
        sub = df_tests[(df_tests["module_id"] == mid) & (df_tests["hpo_term"] == term)]
        if not sub.empty and sub.iloc[0]["significant"]:
            row[term] = -np.log10(sub.iloc[0]["q_value"] + 1e-10)
        else:
            row[term] = 0.0
    pivot_data[f"M{mid}"] = row

df_heatmap = pd.DataFrame(pivot_data).T.fillna(0)

fig, ax = plt.subplots(figsize=(12, max(3, len(valid_modules) * 0.7)))
sns.heatmap(df_heatmap, cmap="Blues", linewidths=0.4, linecolor="lightgray",
            ax=ax, cbar_kws={"label": "−log₁₀(q-value)"})
ax.set_xlabel("HPO Term", fontsize=11)
ax.set_ylabel("Module", fontsize=11)
ax.set_title("Module Phenotypic Signature Heatmap\n(Synthetic data – demonstration only)",
             fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "module_hpo_heatmap.png", dpi=150)
plt.show()
print("Saved module_hpo_heatmap.png")


## § 9 — Summary

In [ ]:
summary_rows = []
for mid in valid_modules:
    mod_idx  = np.where(community_labels == mid)[0]
    mod_genes = [genes[i] for i in mod_idx]
    n_core   = sum(gene_class[g] == "Core"       for g in mod_genes)
    n_peri   = sum(gene_class[g] == "Peripheral" for g in mod_genes)
    n_unst   = sum(gene_class[g] == "Unstable"   for g in mod_genes)
    n_sig    = len(sig[sig["module_id"] == mid])
    summary_rows.append({
        "Module": f"M{mid}", "Size": len(mod_genes),
        "Core": n_core, "Peripheral": n_peri, "Unstable": n_unst,
        "FDR Signatures": n_sig
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
print()
print("Pipeline completed successfully on synthetic data.")
print()
print("─" * 65)
print("  Context (real research pipeline — not reproduced here):")
print("    • Data: hundreds of IRD genes × thousands of HPO terms")
print("    • Lin's IC-based similarity computed via pyhpo + hp.obo")
print("    • Full graph parameter sweep across many kNN and percentile values")
print("    • Stability analysis over many more perturbation runs")
print("    • Multiple Leiden communities with FDR-significant phenotypic signatures")
print("─" * 65)


---
*Part of the [Evolutionary Genomics & Multi-Omics Portfolio](https://github.com/ShalevYaacov/Shalev-Evolutionary-Genomics-Portfolio)*
